In [1]:
import sys
sys.path.append('/mnt/czi-sci-ai/generate-cross-species-secondary/lakshmi/ModelGenerator')

In [2]:
! export CUDA_LAUNCH_BLOCKING=1

In [3]:
import torch
# Clear CUDA cache
torch.cuda.empty_cache()

# Reset CUDA memory statistics
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()

In [4]:
from torch.cuda.amp import autocast  # For mixed precision
from modelgenerator.tasks import Embed
model = Embed.from_config({"model.backbone": "aido_cell_100m"}).eval()
model = model.backbone

# Move model to GPU
model = model.to('cuda')

# Load trained weights
weights = torch.load("/mnt/czi-sci-ai/generate-cross-species-secondary/lakshmi/ModelGenerator/pretrained_weights/pytorch_model.bin", weights_only=True)

# Check weights from the model and the loaded weights
for name, param in model.named_parameters():
    if name in weights:
        assert torch.allclose(param.data, weights[name])

batch_size = 1
seq_len = 19264

# Example input tensors
input_ids = torch.randint(0, 1000, (batch_size, seq_len)).to('cuda')  # Move to GPU
attention_mask = torch.ones((batch_size, seq_len)).to('cuda')  # Move to GPU

collated_batch = {
    "input_ids": input_ids,
    "attention_mask": attention_mask,
}

# Debugging: Check all devices
print("Model device:", next(model.parameters()).device)
for key, value in collated_batch.items():
    print(f"{key} device: {value.device}")

# Forward pass through the Embed wrapper with mixed precision
with autocast():
    logits = model.forward(input_ids=collated_batch["input_ids"], attention_mask=collated_batch["attention_mask"])

# Print the logits shape to verify the output
print("Logits shape:", logits.shape)


/opt/conda/envs/aido/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model device: cuda:0
input_ids device: cuda:0
attention_mask device: cuda:0


/mnt/czi-sci-ai/generate-cross-species-secondary/lakshmi/ModelGenerator/modelgenerator/backbones/backbones.py:655: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(


Logits shape: torch.Size([1, 19264, 640])


In [5]:
model

aido_cell_100m(
  (encoder): CellFoundationModel(
    (gene_embedding): GeneEmbedding(
      (linear1): Linear(in_features=1, out_features=100, bias=True)
      (linear2): Linear(in_features=100, out_features=100, bias=True)
      (leakyReLU): LeakyReLU(negative_slope=0.1)
      (softmax): Softmax(dim=-1)
      (lookup_table): Embedding(100, 640)
      (mask_embedding): Embedding(1, 640)
      (pad_embedding): Embedding(1, 640)
    )
    (position_embedding): Embedding(19266, 640)
    (encoder): CellFoundationEncoder(
      (layer): ModuleList(
        (0-17): 18 x CellFoundationLayer(
          (attention): CellFoundationAttention(
            (ln): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
            (self): BertSelfFlashAttention(
              (query): Linear(in_features=640, out_features=640, bias=False)
              (key): Linear(in_features=640, out_features=640, bias=False)
              (value): Linear(in_features=640, out_features=640, bias=False)
              